# Account-Level Loss Compression — lite

Takes an account-level PLT and a portfolio-level PLT, compresses the account table, reconstructs the
per-account loss metrics, and checks which ones survive.

**The idea in one line:** the worst years are stored exactly, every other year is replaced by a
per-event table of account *shares* of the portfolio loss.

| table | columns |
|---|---|
| `account_plt` | `accnt_no, event_id, year_id, loss_date, loss` |
| `portfolio_plt` | `event_id, year_id, loss_date, loss` |

Both bases are handled:

- **AEP** — annual *sum* of losses in a year
- **OEP** — largest single *occurrence* in a year

They need different reconstruction paths, which is most of what §4 is about.

In [ ]:
import numpy as np, pandas as pd
from scipy import stats, sparse

Q_RETAIN  = 0.98      # store the worst 2% of years exactly
ALPHA     = 0.99      # metrics reported at this level
BLOCK     = 50_000    # occurrences per block in the OEP scan (sets peak memory)
OCC_KEY   = ["year_id", "event_id", "loss_date"]

assert ALPHA >= Q_RETAIN, "a metric level outside the retained region is not covered"
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

## 1 · The input tables

In production these are read from the vendor extract. Here they are simulated: 30 events,
8 accounts, 20,000 trial years, with accounts sharing an event losing together.

Replace this cell with your two `read_parquet` calls. Casting `accnt_no` to `category` and the ids to
`int32` roughly halves memory on a 10M-row table and speeds up every groupby below.

In [ ]:
rng, T_YEARS = np.random.default_rng(0), 20_000

events   = np.arange(1, 31)
rates    = 0.30 * 0.88 ** np.arange(30)
accounts = [f"ACC-{i:02d}" for i in range(1, 9)]
expo     = dict(zip(accounts, [2500, 1800, 1200, 900, 3000, 600, 1500, 2200]))
partic   = {a: events[rng.random(30) < 0.45] for a in accounts}        # event footprints
mdr      = {a: dict(zip(partic[a], np.geomspace(0.002, rng.uniform(0.05, 0.30), len(partic[a]))))
            for a in accounts}

rows = []
for ev, rate in zip(events, rates):
    n   = rng.poisson(rate, T_YEARS); M = int(n.sum())
    yr  = np.repeat(np.arange(T_YEARS), n)
    day = rng.integers(1, 366, M)
    z   = rng.standard_normal(M)                                       # shared shock
    for a in accounts:
        if ev not in mdr[a]:
            continue
        m  = mdr[a][ev]
        u  = stats.norm.cdf(0.6 * z + 0.8 * rng.standard_normal(M))
        nu = m * (1 - m) / (0.5 * m) ** 2 - 1
        rows.append(pd.DataFrame({"accnt_no": a, "event_id": ev, "year_id": yr,
                                  "loss_date": day,
                                  "loss": stats.beta.ppf(u, m * nu, (1 - m) * nu) * expo[a]}))

account_plt   = pd.concat(rows, ignore_index=True)
portfolio_plt = account_plt.groupby(OCC_KEY, as_index=False)["loss"].sum()
del rows

A_IDX = {a: i for i, a in enumerate(accounts)}
N_ACC = len(accounts)
print(f"{len(account_plt):,} account rows | {len(portfolio_plt):,} occurrences | "
      f"{N_ACC} accounts | {T_YEARS:,} years")
print(f"accounts per occurrence {len(account_plt)/len(portfolio_plt):.2f}")
display(account_plt.head())

## 2 · Compress

**Step 1 — the two annual metrics.** `Y[t]` = total loss in year `t` (AEP), `X[t]` = largest single
occurrence in year `t` (OEP).

**Step 2 — pick the years to keep.** The worst `(1−q)·T` by `Y`, **union** the worst `(1−q)·T` by `X`.
The union matters: a year with one enormous event ranks high on OEP but may not on AEP, and a year of
several moderate events ranks high on AEP but not OEP. Ranking is on the **year**, never on
individual occurrences — a bad year can be several moderate events, none individually large.

**Step 3 — split.** Retained years: every account row kept as-is. All other years ("the body"): rows
discarded, replaced by one share vector per event.

The share is `Σ losses to account a ÷ Σ portfolio loss` over all body occurrences of that event —
**sum first, then divide**. Averaging per-row ratios instead would bias against accounts that take a
larger slice of the larger events.

In [ ]:
# --- 1. annual metrics -----------------------------------------------------
Y = np.bincount(portfolio_plt.year_id, weights=portfolio_plt.loss, minlength=T_YEARS)   # AEP
X = (portfolio_plt.groupby("year_id")["loss"].max()
     .reindex(range(T_YEARS), fill_value=0.0).to_numpy())                               # OEP

# --- 2. retained years: union of the AEP and OEP tails --------------------
k        = int(np.ceil((1 - Q_RETAIN) * T_YEARS))
rank     = lambda v, n: np.argsort(-v, kind="stable")[:n]
keep_yrs = np.union1d(rank(Y, k), rank(X, k))
is_kept  = np.zeros(T_YEARS, bool); is_kept[keep_yrs] = True

# --- 3a. retained rows, verbatim ------------------------------------------
tail_store = account_plt[is_kept[account_plt.year_id.to_numpy()]].copy()

# --- 3b. body rows -> one share vector per event --------------------------
body_acct = account_plt[~is_kept[account_plt.year_id.to_numpy()]]
body_port = portfolio_plt[~is_kept[portfolio_plt.year_id.to_numpy()]]

numer  = body_acct.groupby(["event_id", "accnt_no"])["loss"].sum()      # sum first ...
denom  = body_port.groupby("event_id")["loss"].sum()
shares = (numer / denom).unstack(fill_value=0.0).reindex(columns=accounts).fillna(0.0)  # ... then divide

assert np.allclose(shares.sum(axis=1), 1.0), "each event's shares must sum to 1"
print(f"AEP tail {k:,} years | OEP tail {k:,} | union {len(keep_yrs):,} "
      f"({len(keep_yrs)/k:.2f}x a single metric)")
print(f"stored {len(tail_store):,} rows = {100*len(tail_store)/len(account_plt):.1f}% of the "
      f"account table, plus a {shares.shape[0]} x {shares.shape[1]} share table")
display(shares.head())

## 3 · What reconstruction means

For a retained year, read the stored row. For a body occurrence with event `e` and portfolio loss
`L`:

    loss(account a) = share[e, a] × L

Note this is *denser* than the truth: the share vector is non-zero for every account ever touched by
event `e`, so reconstruction assigns a loss to all of them on every occurrence, where the real table
only has rows for the accounts actually hit that time. Check the inflation on your own data before
materialising anything row-level:

In [ ]:
per_occ   = len(account_plt) / len(portfolio_plt)
per_event = account_plt.groupby("event_id")["accnt_no"].nunique().mean()
print(f"accounts per occurrence {per_occ:.2f} | accounts per event {per_event:.2f} "
      f"-> row inflation {per_event/per_occ:.2f}x")
print(f"a row-level reconstruction would be ~{len(portfolio_plt)*per_event:,.0f} rows "
      f"vs {len(account_plt):,} true")
print("\nThe metrics below need per-account ANNUAL values, so the row-level table is never built.")

## 4 · Reconstruct the annual metrics

The two bases need different routes, and this is the part that matters at scale.

**AEP — annual sum.** Summing over occurrences is exactly a matrix product: group body losses by
`(year, event)`, then multiply that sparse matrix by the share table. Output is `T × n_accounts`
directly; nothing occurrence-level is ever materialised.

**OEP — annual max.** A max cannot be recovered from sums, so per-occurrence values are needed. But
they are consumed in **blocks**: build `BLOCK × n_accounts` at a time and fold it into a running max.
Peak memory is `BLOCK × n_accounts × 8` bytes no matter how many occurrences there are — 200 MB at
50,000 × 500. This is the one place blocking is genuinely required.

Retained years are added afterwards straight from the stored rows, so they stay exact.

In [ ]:
# ---------- AEP: annual SUM per account -----------------------------------
AEP = np.zeros((T_YEARS, N_ACC))
by  = body_port.groupby(["year_id", "event_id"], observed=True)["loss"].sum().reset_index()
ec  = shares.index.get_indexer(by.event_id)
assert (ec >= 0).all(), "a body occurrence has an event absent from the share table"
S   = sparse.csr_matrix((by.loss.to_numpy(), (by.year_id.to_numpy(), ec)),
                        shape=(T_YEARS, len(shares)))
AEP += S @ shares.to_numpy()                                     # body, in one product
np.add.at(AEP, (tail_store.year_id.to_numpy(), tail_store.accnt_no.map(A_IDX).to_numpy()),
          tail_store.loss.to_numpy())                            # retained years, exact

# ---------- OEP: annual MAX per account -----------------------------------
OEP  = np.zeros((T_YEARS, N_ACC))
Sarr = shares.to_numpy()
ecb  = shares.index.get_indexer(body_port.event_id)
yrb, lsb = body_port.year_id.to_numpy(), body_port.loss.to_numpy()
for s in range(0, len(body_port), BLOCK):
    sl = slice(s, s + BLOCK)
    np.maximum.at(OEP, yrb[sl], Sarr[ecb[sl]] * lsb[sl][:, None])   # only BLOCK x N_ACC alive
np.maximum.at(OEP, (tail_store.year_id.to_numpy(), tail_store.accnt_no.map(A_IDX).to_numpy()),
              tail_store.loss.to_numpy())                            # retained years, exact

print(f"AEP via sparse product | OEP via {len(range(0, len(body_port), BLOCK))} blocks of "
      f"{BLOCK:,} (peak {BLOCK*N_ACC*8/1e6:.0f} MB)")

In [ ]:
# ---------- the truth, for comparison only --------------------------------
ay, aa, al = (account_plt.year_id.to_numpy(), account_plt.accnt_no.map(A_IDX).to_numpy(),
              account_plt.loss.to_numpy())
AEP_T = np.zeros((T_YEARS, N_ACC)); np.add.at(AEP_T, (ay, aa), al)
OEP_T = np.zeros((T_YEARS, N_ACC)); np.maximum.at(OEP_T, (ay, aa), al)
print(f"AEP totals reconcile: {np.allclose(AEP_T.sum(1), Y)} | "
      f"OEP portfolio max reconciles: {np.allclose(portfolio_plt.groupby('year_id').loss.max().reindex(range(T_YEARS), fill_value=0), X)}")

## 5 · The metrics

- **AAL** — mean annual loss
- **VaR / TVaR** at 99%, standalone: on the account's *own* worst years
- **co-TVaR** — the allocation metric, read in the years where the **portfolio** is worst

co-TVaR is defined differently on the two bases, and the OEP one matters:

- *AEP basis*: the account's annual sum, averaged over the portfolio's worst `Y` years.
- *OEP basis*: the account's loss **on the single largest portfolio occurrence** of each worst `X`
  year — not the account's own annual max. Taking each account's own max would not sum back to the
  portfolio OEP TVaR, because different accounts peak on different occurrences. Reading the one
  occurrence keeps the allocation additive.

In [ ]:
m         = int(round((1 - ALPHA) * T_YEARS))
aep_tail  = rank(Y, m)                          # portfolio's worst years, AEP basis
oep_tail  = rank(X, m)                          # ... OEP basis

# the single largest portfolio occurrence in each OEP-tail year
biggest = (portfolio_plt.loc[portfolio_plt.groupby("year_id")["loss"].idxmax()]
           .set_index("year_id").loc[oep_tail].reset_index())
big_key = pd.MultiIndex.from_frame(biggest[OCC_KEY])

def oep_cotvar(acct_rows):
    hit = acct_rows.set_index(OCC_KEY)
    hit = hit[hit.index.isin(big_key)]
    return hit.groupby("accnt_no")["loss"].sum().reindex(accounts).fillna(0.0).to_numpy() / m

# reconstructed: retained years are stored, so those occurrences are exact; any that fell in
# the body are rebuilt from the share table
in_store = biggest.set_index(OCC_KEY).index.isin(tail_store.set_index(OCC_KEY).index)
rec_occ  = oep_cotvar(tail_store[tail_store.set_index(OCC_KEY).index.isin(big_key)].reset_index(drop=True))
miss     = biggest[~in_store]
if len(miss):
    rec_occ = rec_occ + (shares.to_numpy()[shares.index.get_indexer(miss.event_id)]
                         * miss.loss.to_numpy()[:, None]).sum(0) / m
true_occ = oep_cotvar(account_plt)

def table(A_, O_, occ_):
    sa, so = np.sort(A_, 0), np.sort(O_, 0)
    return pd.DataFrame({"AEP AAL": A_.mean(0),
                         "AEP VaR": sa[-m], "AEP TVaR": sa[-m:].mean(0),
                         "AEP coTVaR": A_[aep_tail].mean(0),
                         "OEP VaR": so[-m], "OEP TVaR": so[-m:].mean(0),
                         "OEP coTVaR": occ_}, index=accounts)

true, recon = table(AEP_T, OEP_T, true_occ), table(AEP, OEP, rec_occ)
err = (recon - true).abs() / true.replace(0, np.nan) * 100
display(pd.concat({"true": true, "reconstructed": recon}, axis=1)
        .swaplevel(axis=1).sort_index(axis=1).round(2))
display(err.round(4).rename(columns=lambda c: c + " err %"))

In [ ]:
print("Portfolio level")
print(f"  AEP TVaR   true {np.sort(Y)[-m:].mean():>12,.2f}   recon {np.sort(AEP.sum(1))[-m:].mean():>12,.2f}")
print(f"  OEP TVaR   true {np.sort(X)[-m:].mean():>12,.2f}   recon {biggest.loss.mean():>12,.2f}")
print(f"  sum AEP coTVaR  true {true['AEP coTVaR'].sum():>10,.2f}   "
      f"recon {recon['AEP coTVaR'].sum():>10,.2f}   (= portfolio AEP TVaR)")
print(f"  sum OEP coTVaR  true {true['OEP coTVaR'].sum():>10,.2f}   "
      f"recon {recon['OEP coTVaR'].sum():>10,.2f}   (= portfolio OEP TVaR)")

print("\nWorst per-account error")
for c in err.columns:
    print(f"  {c:12s} {err[c].max():8.4f}%")

## What survives

| metric | status | why |
|---|---|---|
| **AAL** | exact | shares fitted by summing then dividing, so each account's body total is reproduced identically |
| **AEP co-TVaR** | exact | reads only the portfolio's worst `Y` years, which are stored verbatim |
| **OEP co-TVaR** | exact | reads only the largest occurrence of each worst `X` year, also stored |
| **portfolio AEP / OEP** | exact | the portfolio table is kept whole |
| **per-account VaR / TVaR** | approximate | an account's own worst year is often a quiet year for the portfolio — a body year, where its loss is a share of the total rather than its real loss |

That last row is the trade. The compression is built for capital allocation, which reads the
portfolio's tail — not for per-account return periods, which read each account's own tail.

**OEP degrades more than AEP.** The share vector throws away how an event's split varies from one
occurrence to the next. A sum averages that noise out; a max is drawn to the occurrence where the
real split happened to favour an account, and that is exactly the variation the reconstruction has
flattened.

Three rules that keep it honest:

- `ALPHA >= Q_RETAIN`. A metric level outside the retained region is not covered.
- Rank on the **year**, not on individual occurrences.
- If OEP is reported, retain the **union** of the AEP and OEP tails.